# MLIP Active-Learning Tutorial

> New to ALF? Start with the [Installation Guide](https://instadeepai.github.io/alf/installation.html).

This tutorial shows how the ALF `MLIPModel` (a MACE machine-learned interatomic
potential) is used in an **offline (pool-based) active-learning loop**. Our use case: starting from
a pretrained foundation model and **finetuning** it into an accurate force field for a single
organic molecule, while spending as few expensive labels as possible.

### How this differs from the design tutorials

The protein-design tutorials *maximise a fitness*. Here we do the opposite kind of active
learning: we **minimise model error using as few expensive labels as possible**. Each label is an
expensive quantum-chemistry calculation (DFT). We work from a fixed pool of DFT-labelled aspirin
configurations and let the model decide which ones are worth "paying" to reveal — choosing the
configurations where its committee of models *disagrees most* (often the most informative ones,
though, as we'll see, not always).

### Experiment overview

1. Download a pretrained MACE organics model and a pool of DFT-labelled aspirin configurations.
2. Finetune a committee (ensemble) of models on a small seed set and use their disagreement to
   estimate prediction uncertainty.
3. Each round: score the remaining candidate pool, acquire the most uncertain configurations,
   reveal their DFT labels, and finetune again.
4. Compare an uncertainty-driven acquisition against a random baseline, and see why acquisition
   choice matters.

### Framework Components

1. **Dataset** (`AspirinDataset`, defined below): loads DFT-labelled aspirin configurations and
   splits them into seed-train / validation / test / **candidate pool**. The pool holds the
   unlabelled configurations active learning chooses from.
2. **Surrogate Model** ([`MLIPModel`](https://instadeepai.github.io/alf/api/alf_tools/models/)) wrapped in an [`EnsembleWrapper`](https://instadeepai.github.io/alf/api/alf_tools/models/) committee: each member finetunes the pretrained MACE model; their disagreement is our uncertainty.
3. **Search Strategy** ([`DatasetSearch`](https://instadeepai.github.io/alf/api/alf_core/optimizer/search/)): serves the remaining candidate pool each round.
4. **Acquisition Function** (`MaxVariance`, defined below): selects the configurations where the committee disagrees most (highest prediction variance). `RandomAcquisition` is the baseline for comparison.
5. **Optimizer** ([`Optimizer`](https://instadeepai.github.io/alf/api/alf_core/optimizer/optimizer/)): handles the ask/tell cycle.
6. **Oracle** ([`Oracle`](https://instadeepai.github.io/alf/api/alf_core/oracle/) with the dataset as scorer): reveals the precomputed DFT energy (and forces) for an acquired configuration via `dataset.query`.
7. **Task** ([`DesignTask`](https://instadeepai.github.io/alf/api/alf_core/tasks/design_task/)): orchestrates the active-learning loop.

## Setup

These tutorials are written for **dev mode** — running from a local clone of the ALF repository.
The MLIP tutorial needs the `mlip` package (the MACE force field), which ALF exposes through the
optional `mlip` extra (mirrored as the `mlip` dependency group in `tutorials/pyproject.toml`). From
the `tutorials/` directory, sync that group so the extra is installed alongside the tutorial
dependencies:

```bash
uv sync --group mlip   # installs alf_core, alf_tools[mlip] and tutorial deps (CPU PyTorch by default)
```

Register the environment as a Jupyter kernel, then select the `alf` kernel in this notebook:

```bash
uv run ipython kernel install --user --env VIRTUAL_ENV "$(pwd)/.venv" --name=alf
```

`uv sync` installs the **CPU** build of PyTorch by default. For GPU acceleration, see the
[GPU support section of the Installation Guide](https://instadeepai.github.io/alf/installation.html#gpu-support).

**Not running from a clone?** If you installed ALF with `pip`, run the optional cell below to
install this tutorial's dependencies into the current kernel, then restart the kernel.

In [ ]:
# Optional — only needed if you are NOT running from a cloned repo via `uv sync`.
# Installs ALF and this tutorial's dependencies into the current kernel, then restart the kernel.
# %pip install "alf_tools[mlip]" matplotlib pandas huggingface_hub

### Step 0: Download the Model and Dataset

The cell below downloads the pretrained MACE organics model and the public **rMD17 aspirin** dataset
(both from Hugging Face, public, no credentials) into the local cache. The aspirin configurations
carry DFT energies and forces, so no live quantum-chemistry engine is needed.

In [ ]:
from pathlib import Path

# Pretrained MACE organics foundation model -> alf models dir, so the MLIPModel loader
# finds it locally and skips its (private) S3 fallback.
import alf_tools.models.utils.mlip_utils as mlip_utils  # noqa: E402
from huggingface_hub import hf_hub_download, snapshot_download

models_dir = mlip_utils._MODELS_DIR
models_dir.mkdir(parents=True, exist_ok=True)
hf_hub_download(
    repo_id="InstaDeepAI/mlip_models_organics_v2",
    filename="mace_organics_02.zip",
    local_dir=str(models_dir),
)

# Public rMD17 aspirin dataset (DFT energies + forces) from the mlip tutorials collection.
DATA_DIR = Path("data/aspirin")
snapshot_download(
    repo_id="InstaDeepAI/MLIP-tutorials",
    allow_patterns="training/rmd17_aspirin_*",
    local_dir=str(DATA_DIR),
)
ASPIRIN_TRAIN_XYZ = DATA_DIR / "training" / "rmd17_aspirin_train.xyz"

print(f"✅ Pretrained model present at {models_dir / 'mace_organics_02.zip'}")
print(f"✅ rMD17 aspirin dataset present at {ASPIRIN_TRAIN_XYZ}")

### Step 1: Import Required Libraries

In [ ]:
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from alf_core import (
    AcquisitionFunction,
    Candidate,
    DatasetSearch,
    DesignTask,
    FileStateLogger,
    LabelledCandidates,
    Optimizer,
    Oracle,
    State,
    Surrogate,
    TerminalStateLogger,
)
from alf_core.dataclasses.candidate import Modality
from alf_core.dataset.base_dataset import BaseDataset, BaseDatasetConfig
from alf_core.utils.enums import ProblemType
from alf_tools.models.ensemble import EnsembleWrapper, EnsembleWrapperConfig, SubsampleConfig
from alf_tools.models.mlip import MLIPModel, MLIPModelConfig, MLIPTrainConfig
from ase.io import read as ase_read

print("✅ All imports successful!")

### Step 2: Load the rMD17 Aspirin Dataset

We load configurations of **aspirin** — sampled from molecular-dynamics trajectories in the revised
MD17 (rMD17) dataset — from the public mlip tutorials collection. Each configuration carries a DFT
energy (our regression label) and forces (stored in each candidate's `features` so the MACE model
can train on them). We subsample `N_TOTAL` configurations and let ALF split them into a small
**seed-train** set, a **validation** set, a held-out **test** set, and a **candidate pool**. Active
learning then reveals labels from the pool a few configurations at a time.

In [ ]:
DATA_SEED = 51505
N_TOTAL = 150  # subsample of rMD17 aspirin configs to keep the notebook CPU-fast


def load_aspirin(
    xyz_path: Path | str, max_configs: int | None = None, seed: int = 0
) -> LabelledCandidates:
    """Read an extxyz file of aspirin configurations into `LabelledCandidates`.

    Energy (eV) is the regression label; forces (N, 3) are stored in each
    candidate's `features` so the MACE model can train on them.
    """
    atoms_list = ase_read(str(xyz_path), index=":")
    if max_configs is not None and len(atoms_list) > max_configs:
        rng = np.random.default_rng(seed)
        idx = rng.choice(len(atoms_list), size=max_configs, replace=False)
        atoms_list = [atoms_list[i] for i in idx]
    candidates, labels = [], []
    for atoms in atoms_list:
        energy = atoms.get_potential_energy()
        forces = atoms.get_forces()
        candidates.append(
            Candidate(data=atoms, modality=Modality.STRUCTURE, features={"forces": forces})
        )
        labels.append(energy)
    return LabelledCandidates(candidates=candidates, labels=np.asarray(labels))


class AspirinDataset(BaseDataset):
    """Pre-labelled aspirin conformers; alf splits them into seed-train / val / test / pool."""

    def __init__(self, config: BaseDatasetConfig, data: LabelledCandidates):
        """Store the pre-built labelled data; `setup()` performs the split."""
        super().__init__(config)
        self._data = data

    def load_dataset(self) -> LabelledCandidates:
        """Return the pre-built `LabelledCandidates` (energy labels, forces in features)."""
        return self._data


def make_aspirin_dataset(
    data: LabelledCandidates | None = None, seed: int = DATA_SEED
) -> AspirinDataset:
    """Build and set up an `AspirinDataset` with the CPU-tiny split used in this tutorial.

    Pass `data` to reuse an already-loaded set of configurations; otherwise the
    configurations are loaded fresh from `ASPIRIN_TRAIN_XYZ`.
    """
    if data is None:
        data = load_aspirin(ASPIRIN_TRAIN_XYZ, max_configs=N_TOTAL, seed=seed)
    ds = AspirinDataset(
        BaseDatasetConfig(
            name="rmd17_aspirin",
            modality=Modality.STRUCTURE,
            seed=seed,
            train_ratio=0.12,
            validation_frac=0.2,
            test_ratio=0.25,
            split_type="random",
            problem_type=ProblemType.REGRESSION,
        ),
        data,
    )
    ds.setup()
    return ds


data = load_aspirin(ASPIRIN_TRAIN_XYZ, max_configs=N_TOTAL, seed=DATA_SEED)

# Fail fast if aspirin contains any element the pretrained organics model never saw.
from mlip.models.mace.network import Mace  # noqa: E402

_pretrained_ff = mlip_utils._load_model_from_zip(Mace, "mace_organics_02.zip")
_ztable = set(_pretrained_ff.dataset_info.atomic_energies_map.keys())
_missing = {int(n) for c in data.candidates for n in c.data.numbers} - _ztable
if _missing:
    raise ValueError(f"aspirin contains elements outside the pretrained z-table: {_missing}")

dataset = make_aspirin_dataset(data)
print(dataset)
print(
    f"✅ rMD17 aspirin dataset ready — pool={len(dataset.candidate_pool)}, "
    f"test={len(dataset.test_dataset)}"
)

### Step 3: The Oracle (precomputed DFT)

The oracle is the expensive ground-truth evaluator. Here each aspirin configuration already has a
DFT energy, so the oracle simply **reveals** that label on demand: `Oracle(scorer=dataset)` calls
`dataset.query(...)` to look up the energy for an acquired configuration. This mirrors production
active learning, where labelling is costly and is therefore spent sparingly on the most informative
structures.

In [ ]:
# Offline oracle: reveal the precomputed DFT label for an acquired configuration.
# In production this is an expensive DFT calculation; here the labels already exist in
# the dataset, so the oracle is a lookup (`dataset.query`) keyed by candidate identity.
oracle = Oracle(scorer=dataset)
print("✅ Oracle ready (offline lookup of precomputed DFT labels)!")

### Step 4: Search and Acquisition

`DatasetSearch` serves the remaining **candidate pool** — the aspirin configurations not yet
acquired. `MaxVariance` (used next) scores them by committee disagreement, picking the
configurations the ensemble is least sure about. `RandomAcquisition` is the baseline that ignores
the model and picks at random, so we can show that uncertainty-driven selection actually helps.

In [ ]:
class MaxVariance(AcquisitionFunction):
    """Select conformers where the committee disagrees most (highest prediction variance).

    This is query-by-committee uncertainty sampling: the structures the ensemble is least
    certain about are the most informative to label next. It is the natural acquisition for
    *reducing model error* (our goal), as opposed to maximising a property.
    """

    def __call__(self, search_candidates: list[Candidate], state: State) -> LabelledCandidates:
        """Score each candidate by its committee variance (higher = more uncertain)."""
        predictions = state.surrogate.predict(search_candidates)
        if predictions.variances is None:
            raise ValueError("MaxVariance requires committee variances; use an ensemble surrogate.")
        return LabelledCandidates(candidates=search_candidates, labels=predictions.variances)


class RandomAcquisition(AcquisitionFunction):
    """Baseline acquisition that scores candidates uniformly at random."""

    def __init__(self, seed: int = 0):
        """Seed the RNG used to score candidates."""
        self.seed = seed

    def __call__(self, search_candidates: list[Candidate], state: State) -> LabelledCandidates:
        """Assign each candidate a random acquisition value."""
        rng = np.random.default_rng(self.seed)
        scores = rng.random(len(search_candidates))
        return LabelledCandidates(candidates=search_candidates, labels=scores)


# Offline pool search: serve the remaining (unlabelled) candidate pool each round.
search_fn = DatasetSearch()
print("✅ Search strategy initialised (offline candidate pool)!")

### Step 5: The Surrogate — a Committee of Finetuned MACE Models

Our surrogate is an ensemble (committee) of `MLIPModel`s. Each member finetunes the **same**
pretrained MACE model, but on a different bootstrap resample of the training data, so the members
end up slightly different. Where they disagree most, the model is most uncertain — exactly the
conformers worth labelling. `EnsembleWrapper.predict` returns both the mean energy and the
variance across members.

In [ ]:
def mlip_factory(seed: int) -> MLIPModel:
    """Build a finetuning MLIPModel (from the pretrained MACE model) with the given seed."""
    return MLIPModel(
        model_config=MLIPModelConfig(model_path="mace_organics_02.zip"),
        train_config=MLIPTrainConfig(epochs=8, batch_size=2, learning_rate=1e-3),
        seed=seed,
    )


def make_surrogate(n_members: int = 2) -> Surrogate:
    """Build a committee surrogate of finetuned MACE models (1 member = no uncertainty)."""
    return Surrogate(
        model=EnsembleWrapper(
            model_factory=mlip_factory,
            config=EnsembleWrapperConfig(
                base_seed=0,
                n_members=n_members,
                subsample=SubsampleConfig(fraction=1.0, replace=True),
            ),
        )
    )


surrogate = make_surrogate(n_members=2)
print("✅ Surrogate committee initialised (2 finetuned MACE members)!")

### Step 6: Optimizer and Design Task

`MaxVariance` ranks candidates purely by the committee's prediction variance, so each round we
label the pool configurations the ensemble disagrees on most — classic query-by-committee active
learning. The `Optimizer` combines it with the `DatasetSearch` over the candidate pool; `DesignTask`
runs the rounds.

In [ ]:
acquisition_fn = MaxVariance()
optimizer = Optimizer(acquisition_fn=acquisition_fn, search_fn=search_fn)

num_acq_rounds = 3
acq_batch_size = 3
task = DesignTask(num_acq_rounds=num_acq_rounds, acq_batch_size=acq_batch_size)
print(f"✅ Optimizer + DesignTask ready ({num_acq_rounds} rounds × {acq_batch_size} labels)!")

### Step 7: Run the Active-Learning Experiment (uncertainty-driven)

Each round: take the remaining candidate pool, score it with the committee, acquire the most
uncertain configurations, reveal their DFT labels, finetune the committee, and evaluate on the
held-out test set.

In [ ]:
import logging

logging.basicConfig(level=logging.WARNING)  # keep MACE output quiet in the notebook

maxvar_path = Path("results/mlip_design/")
if maxvar_path.exists():
    shutil.rmtree(maxvar_path)
loggers = [TerminalStateLogger(), FileStateLogger(output_path=maxvar_path)]

state = task.setup(dataset=dataset, surrogate=surrogate)
print("🚀 Running uncertainty-driven active learning...")
task.run(state, state_loggers=loggers, optimizer=optimizer, oracle=oracle)
print("✅ Uncertainty-driven experiment completed!")

### Step 8: Random-Acquisition Baseline

To show that uncertainty-driven selection helps, we run the identical loop over the same aspirin
splits but pick pool configurations at random. The baseline needs no committee, so it uses a single
finetuned model (and its own dataset-bound oracle).

In [ ]:
# Rebuild a fresh dataset so the baseline starts from the same initial splits.
dataset_random = make_aspirin_dataset()
random_oracle = Oracle(scorer=dataset_random)  # oracle is bound to THIS dataset's labels

random_optimizer = Optimizer(acquisition_fn=RandomAcquisition(seed=0), search_fn=search_fn)
random_surrogate = make_surrogate(n_members=1)  # no uncertainty needed for random
random_task = DesignTask(num_acq_rounds=num_acq_rounds, acq_batch_size=acq_batch_size)

random_path = Path("results/mlip_design_random/")
if random_path.exists():
    shutil.rmtree(random_path)
random_loggers = [TerminalStateLogger(), FileStateLogger(output_path=random_path)]

random_state = random_task.setup(dataset=dataset_random, surrogate=random_surrogate)
print("🚀 Running random-acquisition baseline...")
random_task.run(
    random_state, state_loggers=random_loggers, optimizer=random_optimizer, oracle=random_oracle
)
print("✅ Baseline completed!")

### Step 9: Results

The headline metric is the **learning curve**: test-set energy error against the number of labels
acquired. If uncertainty-driven acquisition works, its error drops faster than the random
baseline's. We also show a parity plot of the finetuned baseline model's predicted vs. DFT energies
on the test set.

In [ ]:
maxvar_metrics = pd.read_csv("results/mlip_design/metrics.csv")
rnd_metrics = pd.read_csv("results/mlip_design_random/metrics.csv")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("MLIP Active Learning: uncertainty-driven vs random", fontsize=15, fontweight="bold")

for df, label, color in [
    (maxvar_metrics, "MaxVariance committee", "#e74c3c"),
    (rnd_metrics, "Random", "#3498db"),
]:
    axes[0].plot(
        df["dataset/num_train"], df["surrogate/test_mse"], marker="o", label=label, color=color
    )
    axes[1].plot(
        df["dataset/num_train"], df["surrogate/test_spearman"], marker="o", label=label, color=color
    )

axes[0].set_xlabel("Number of labelled structures")
axes[0].set_ylabel("Test energy MSE (eV²)")
axes[0].set_title("Learning curve (lower is better)")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].set_xlabel("Number of labelled structures")
axes[1].set_ylabel("Test Spearman")
axes[1].set_title("Rank correlation (higher is better)")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

**Reading the learning curves.** The left panel is the headline result: test-set energy MSE (eV²,
lower is better) plotted against the number of labelled structures, which grows by `acq_batch_size`
each round. The right panel tracks rank correlation (Spearman, higher is better) over the same
budget. Both lines start at the seed-trained committee and step rightward as each round reveals more
DFT labels. If uncertainty-driven acquisition is helping, the red `MaxVariance` curve should fall
(and its Spearman rise) faster than the blue `Random` baseline.

On a problem this small — 150 configurations, a 2-member committee, and only a few rounds — expect
the two curves to be close and a little noisy: bootstrap resampling of a handful of structures gives
only a coarse uncertainty signal, so small gaps are not decisive. The point here is that the
end-to-end loop runs and the error trends downward as labels are spent; we return to *why*
max-variance need not win on a toy problem in the conclusion.

Next we check the model's absolute accuracy with a parity plot.

In [ ]:
# Parity on the held-out test set. We use the (stable) single-model random baseline for a
# clean predicted-vs-DFT comparison.
test_cands = dataset_random.test_dataset.candidates
test_true = dataset_random.test_dataset.labels
test_pred = random_surrogate.predict(test_cands).means

plt.figure(figsize=(6, 6))
plt.scatter(test_true, test_pred, alpha=0.7, color="#9b59b6")
lims = [min(test_true.min(), test_pred.min()), max(test_true.max(), test_pred.max())]
plt.plot(lims, lims, "k--", alpha=0.5)
plt.xlabel("DFT energy (eV)")
plt.ylabel("Predicted energy (eV)")
plt.title("Finetuned MLIP: predicted vs. DFT energy (test set)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Reading the parity plot.** Each point is a held-out test configuration: its DFT energy on the
x-axis against the finetuned model's prediction on the y-axis, with the dashed line marking perfect
agreement. Points hugging the diagonal mean the finetuned MACE model reproduces DFT energies
faithfully. Because rMD17 aspirin energies span only a narrow window (~1–2 eV about the mean), a
healthy result is a tight cluster along the diagonal; a constant vertical offset would point to a
residual reference-energy (E0) mismatch rather than poorly learned energy *differences*, which are
what matter for ranking and for forces.

## Conclusion

We built an offline (pool-based) active-learning loop for a machine-learned interatomic potential
with ALF: finetune a pretrained MACE committee on a few DFT-labelled aspirin configurations,
estimate uncertainty from committee disagreement, acquire more configurations from a candidate pool,
and repeat. Every component is a standard ALF abstraction — `AspirinDataset`, the dataset-lookup
`Oracle`, `DatasetSearch`, the `EnsembleWrapper` surrogate, `MaxVariance`/`RandomAcquisition`,
`Optimizer`, and `DesignTask`.

This example is deliberately tiny and fast, so treat the learning curves as illustrative rather
than conclusive. On a problem this small, uncertainty (max-variance) sampling does not reliably beat
random selection — a useful reminder that the acquisition strategy must be matched to the problem
and validated, not assumed. Query-by-committee active learning is the workhorse of *production* MLIP
training, where it selects from large, diverse pools of physically meaningful structures.

### Next steps

- Scale up: a larger seed set, a bigger candidate pool, more acquisition rounds, and a larger
  held-out test set (rMD17 aspirin ships 1200/900/900 train/val/test configurations).
- Try a different molecule or train from scratch (`model_path=None`) for chemistries with no
  compatible foundation model.
- Honour rMD17's curated train/test split (or a temperature split) instead of re-splitting one file.

**Happy modelling!** ⚛️🔬✨

In [ ]:
for p in [Path("results/mlip_design/"), Path("results/mlip_design_random/")]:
    if p.exists():
        shutil.rmtree(p)
print("✅ Results directories cleaned up!")